# Social E-commerce Purchase Prediction

This notebook is the analysis companion for the productionized training code in `training_pipeline.py`.

It now uses a single source of truth for:
- task definitions (`listing_time` vs `session_time`)
- feature grouping and leakage control
- split strategy comparison (stratified, time-based, group-based)
- calibrated model training
- persona clustering and profiling

The notebook is intentionally thin: it orchestrates and inspects the reusable pipeline instead of re-implementing the old modeling logic inline.

## 1. Setup

In [ ]:
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from training_pipeline import (
    ARTIFACT_PATH,
    DATA_PATH,
    METRICS_PATH,
    PROCESSED_PATH,
    BEHAVIORAL_SESSION_FEATURES,
    FORBIDDEN_LEAKY_FEATURES,
    LISTING_FEATURES,
    PERSONA_CLUSTER_FEATURES,
    SELLER_CONTROLLABLE,
    SESSION_FEATURES,
    STATIC_USER_CONTEXT,
    load_dataset,
    main as run_training_pipeline,
)

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

## 2. Task Definition And Feature Policy

In [ ]:
feature_policy = pd.DataFrame(
    [
        ('seller_controllable', ', '.join(SELLER_CONTROLLABLE)),
        ('static_user_context', ', '.join(STATIC_USER_CONTEXT)),
        ('behavioral_session_features', ', '.join(BEHAVIORAL_SESSION_FEATURES)),
        ('forbidden_leaky_features', ', '.join(FORBIDDEN_LEAKY_FEATURES)),
    ],
    columns=['group', 'features'],
)
feature_policy

In [ ]:
task_summary = pd.DataFrame(
    [
        {
            'task': 'listing_time',
            'purpose': 'Predict conversion using only features available when the listing is created.',
            'features': len(LISTING_FEATURES),
            'feature_list': ', '.join(LISTING_FEATURES),
        },
        {
            'task': 'session_time',
            'purpose': 'Predict conversion during an active session using behavioral context.',
            'features': len(SESSION_FEATURES),
            'feature_list': ', '.join(SESSION_FEATURES),
        },
    ]
)
task_summary

## 3. Raw Dataset Snapshot

In [ ]:
df_raw = load_dataset()
print(df_raw.shape)
df_raw.head()

## 4. Run Or Refresh The Unified Pipeline

In [ ]:
# Uncomment the next line to regenerate all model artifacts from the latest code.
# run_training_pipeline()

## 5. Load Generated Artifacts

In [ ]:
assert ARTIFACT_PATH.exists(), 'Run `run_training_pipeline()` first to create model_artifacts.pkl'
assert PROCESSED_PATH.exists(), 'Run `run_training_pipeline()` first to create processed_data.csv'
assert METRICS_PATH.exists(), 'Run `run_training_pipeline()` first to create model_metrics.json'

with ARTIFACT_PATH.open('rb') as f:
    artifacts = pickle.load(f)

with METRICS_PATH.open() as f:
    metrics = json.load(f)

df_processed = pd.read_csv(PROCESSED_PATH)
sorted(artifacts.keys())

## 6. Compare Listing-Time And Session-Time Models

In [ ]:
comparison_df = artifacts['comparison_df'].copy()
comparison_df

In [ ]:
plot_df = comparison_df.melt(
    id_vars=['task', 'split'],
    value_vars=['raw_roc_auc', 'calibrated_roc_auc', 'raw_f1', 'calibrated_f1', 'tuned_f1'],
    var_name='metric',
    value_name='value',
)

plt.figure(figsize=(12, 5))
sns.barplot(data=plot_df, x='split', y='value', hue='task', ci=None)
plt.title('Task Comparison Across Split Strategies')
plt.ylabel('Metric Value')
plt.xlabel('Split Strategy')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 7. Inspect Calibration Benefit

In [ ]:
brier_view = comparison_df[
    ['task', 'split', 'raw_brier', 'calibrated_brier']
].copy()
brier_view['brier_improvement'] = brier_view['raw_brier'] - brier_view['calibrated_brier']
brier_view.sort_values(['task', 'split'])

In [ ]:
threshold_view = comparison_df[
    ['task', 'split', 'calibrated_f1', 'tuned_f1', 'tuned_precision', 'tuned_recall', 'tuned_positive_rate', 'tuned_threshold']
].copy()
threshold_view['f1_gain_vs_0_5'] = threshold_view['tuned_f1'] - threshold_view['calibrated_f1']
threshold_view.sort_values(['task', 'split'])

## 8. Persona Clustering Diagnostics

In [ ]:
cluster_eval = pd.DataFrame(artifacts['clustering']['evaluations'])
cluster_eval = cluster_eval.sort_values(
    ['algorithm', 'feature_set', 'use_log', 'use_pca', 'k'],
    na_position='last'
).reset_index(drop=True)
cluster_eval

In [ ]:
selected_cluster_cfg = {
    'algorithm': artifacts['clustering']['algorithm'],
    'feature_set_name': artifacts['clustering']['feature_set_name'],
    'use_log': artifacts['clustering']['use_log'],
    'use_pca': artifacts['clustering']['use_pca'],
    'selected_features': ', '.join(artifacts['clustering']['selected_features']),
}
pd.DataFrame([selected_cluster_cfg])

In [ ]:
kmeans_eval = cluster_eval[cluster_eval['algorithm'] == 'kmeans'].copy()
kmeans_eval['pipeline_variant'] = kmeans_eval.apply(
    lambda row: f"{row['feature_set']} | log={row['use_log']} | pca={row['use_pca']}",
    axis=1,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.lineplot(data=kmeans_eval, x='k', y='silhouette', hue='pipeline_variant', marker='o', ax=axes[0])
axes[0].set_title('K-Means Silhouette By Feature Set / Transform')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Silhouette')
axes[0].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title='Variant')

sns.lineplot(data=kmeans_eval, x='k', y='size_balance', hue='pipeline_variant', marker='o', ax=axes[1])
axes[1].set_title('K-Means Cluster Size Balance')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Min Cluster Share / Max Cluster Share')
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title='Variant')

plt.tight_layout()
plt.show()

In [ ]:
summary_cols = [
    'algorithm', 'feature_set', 'use_log', 'use_pca', 'k',
    'silhouette', 'calinski_harabasz', 'davies_bouldin', 'size_balance', 'cluster_count'
]
cluster_eval[summary_cols].sort_values(
    ['silhouette', 'size_balance'], ascending=[False, False]
).head(15)

In [ ]:
hdbscan_eval = cluster_eval[cluster_eval['algorithm'] == 'hdbscan'].copy()
if len(hdbscan_eval):
    hdbscan_eval[['feature_set', 'use_log', 'silhouette', 'size_balance', 'cluster_count', 'noise_rate']].sort_values(
        ['silhouette', 'size_balance'], ascending=[False, False]
    )
else:
    print('HDBSCAN was not available in this run.')

In [ ]:
top_variants = cluster_eval.copy()
top_variants['variant'] = top_variants.apply(
    lambda row: f"{row['algorithm']} | {row['feature_set']} | log={row['use_log']} | pca={row['use_pca']} | k={row['k']}",
    axis=1,
)
top_variants = top_variants.sort_values(['silhouette', 'size_balance'], ascending=[False, False]).head(10)

plt.figure(figsize=(14, 6))
sns.barplot(data=top_variants, y='variant', x='silhouette', palette='viridis')
plt.title('Top Clustering Candidates By Silhouette')
plt.xlabel('Silhouette')
plt.ylabel('Candidate')
plt.tight_layout()
plt.show()

## 9. Cluster Profiles And Strategies

In [ ]:
cluster_profiles = artifacts['cluster_profiles'].copy()
cluster_profiles['segment_name'] = cluster_profiles.index.map(artifacts['cluster_names'])
cluster_profiles['cluster_weight'] = cluster_profiles['count'] / cluster_profiles['count'].sum()
cluster_profiles

In [ ]:
cluster_strategy_rows = []
for cluster_id, name in artifacts['cluster_names'].items():
    for strategy in artifacts['clustering']['strategies'][cluster_id]:
        cluster_strategy_rows.append({
            'cluster_id': cluster_id,
            'segment_name': name,
            'strategy': strategy,
        })

pd.DataFrame(cluster_strategy_rows)

## 10. Processed Dataset Snapshot

In [ ]:
print(df_processed.shape)
df_processed.head()

## 11. Artifact Summary

This notebook no longer owns the training logic.

The authoritative pipeline now lives in:
- `training_pipeline.py` for feature policy, training, calibration, and clustering
- `model_artifacts.pkl` for runtime consumption by the app/backend
- `model_metrics.json` for compact evaluation summaries

That keeps the notebook, backend, and deployment artifacts aligned.